## Problem Statement

Global CO₂ emissions have shown a significant upward trend over time, especially after industrialization. Understanding how emissions have evolved and forecasting future values is important for environmental analysis and planning.

This project uses historical CO₂ emissions data where:
- The dataset contains **Year** (mostly in decade intervals, with recent yearly values like 2022 and 2023)
- **Emissions** are measured in **million metric tons**

The objectives of this project are:
- Analyze historical emission trends and growth patterns
- Understand how emissions have changed over time
- Predict future CO₂ emissions up to the year **2060**

This is treated as a **time series forecasting problem**, where past values are used to predict future emissions.

## Solution Overview

To solve this time series problem, the data was transformed into a supervised learning format and machine learning models were applied.

### Approach

1. Data Preprocessing
- Cleaned missing and inconsistent values
- Applied log transformation to stabilize variance
- Scaled the year feature

2. Feature Engineering
- Created lag features:
  - lag1 (previous emission)
  - lag2 (two previous emission)
- Created growth features:
  - growth_rate
  - growth_rate_log

3. Exploratory Data Analysis (EDA)
- Analyzed historical trends
- Studied emission growth patterns
- Visualized actual vs predicted values

4. Modeling
- Linear Regression (for trend-based learning)
- Random Forest Regression (for non-linear patterns)

5. Forecasting
- Used **recursive forecasting**
- Updated lag values step-by-step for future predictions

6. Evaluation
- Compared actual vs predicted values
- Analyzed percentage change in emissions

### Goal

To estimate future CO₂ emissions up to **2060** based on historical data.

#### Import Required libraries 

In [ ]:
import pandas as pd
import numpy as np
import kagglehub

import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score

#### Load Dataset


In [ ]:

path = kagglehub.dataset_download("patricklford/global-co-emissions")

print("Path to dataset files:", path)
df = pd.read_csv(f"{path}\\GlobalCO2Emissions.csv")

#### Initial Data Exploration

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

#### Feature Engineering

In [ ]:
df["Emissions_log"] = np.log(df["Emissions"]) 

df["lag1"] = df["Emissions"].shift(1)
df["lag2"] = df["Emissions"].shift(2)
df["lag1_log"] = np.log(df["lag1"])
df["lag2_log"] = np.log(df["lag2"])

df["growth_rate"] = df["lag1"] - df["lag2"]
df["growth_rate_log"] = df["lag1_log"] - df["lag2_log"]

In [ ]:
df["percent_change"] = df["Emissions"].pct_change() * 100

In [ ]:
df['acceleration'] = df['percent_change'].diff()

In [ ]:
def access_trend(x):
    if x > 0:
        return "Increasing"
    elif x < 0:
        return "Decreasing"
    else:
        return "Stable"
df["trend"] = df["acceleration"].apply(access_trend)


In [ ]:
df["emission_trend"] = df["Emissions"].diff()

In [ ]:
df = df.dropna()

In [ ]:
df.tail()

#### EDA

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["Year"], df["Emissions"], marker='o' , label="Emissions" , color= "Black"
          , linestyle="-" , linewidth=2 , markersize=6 , markerfacecolor="Red" , markeredgecolor="white" , markeredgewidth =1)
plt.title("Global CO2 Emissions Over Time", color = "black" , fontsize = 14 , fontweight = "bold" , fontfamily = "serif")
plt.xlabel("Year")
plt.ylabel("Emissions (Million Metric Tons)")
plt.grid(True)
plt.legend()
plt.show() 

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["Year"][3:] , df["acceleration"][3:], marker='o' , label="Percent Change in Acceleration" , color= "black" , linestyle = "-" ,
         linewidth = 2 , markersize = 6 , markerfacecolor = "red" , markeredgecolor = "white" , markeredgewidth = 1)
plt.title("Acceleration of Global CO2 Emissions" , color = "black" , fontsize = 14 , fontweight = "bold" , fontfamily = "serif")
plt.xlabel("Year")
plt.ylabel("Acceleration (%)")
plt.grid()
plt.legend(loc = "best")
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["Year"] , df["percent_change"] , marker = "o" , label = "Percent Change" , color ="black" ,
          linestyle = "-" , linewidth = 2 , markersize = 6 , markerfacecolor = "red" , markeredgecolor = "white" , markeredgewidth = 1)
plt.title("Percent Change in Global CO2 Emissions", color = "black" , fontsize = 14 , fontweight = "bold" , fontfamily = "serif")
plt.xlabel("Year")
plt.ylabel("Percent Change (%)")
plt.grid()
plt.legend(loc = "best")
plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
sns.barplot(data = df , x = df["Year"] , y = "Emissions" , hue = "trend" , palette = {"Increasing": "red", "Decreasing": "green", "Stable": "blue"}, alpha = 0.8, edgecolor = "black" , linewidth = 1)
plt.title("Global CO2 Emissions Trend", color = "black" , fontsize = 14 , fontweight = "bold" , fontfamily = "serif")
plt.xlabel("Year")
plt.ylabel("Emissions (Million Metric Tons)")
plt.grid(axis = "y")
plt.legend(title = "Trend")
plt.show()


In [ ]:
df.plot(x='Year', y='emission_trend', kind='bar' , figsize=(12, 4) , color = "black" , label = "Emission Trend" , edgecolor = "red" , linewidth = 1 , ax=plt.gca())
plt.title("Yearly Change in Global CO2 Emissions", color = "black" , fontsize = 14 , fontweight = "bold" , fontfamily = "serif")
plt.xlabel("Year")
plt.ylabel("Change in Emissions (Million Metric Tons)")
plt.grid(axis='y')
plt.show()

#### Data Preprocessing


In [ ]:
df = df[df["Year"] > 1900]

In [ ]:
min_year = df["Year"].min()
df["Year_scaled"] = df["Year"] - min_year

In [ ]:
df = df.dropna()

#### Feature Split


In [ ]:
X = df[["lag1_log", "growth_rate_log" , "Year_scaled"]]
y = df["Emissions_log"]

#### Baseline Model

In [ ]:
df["baseline_log"] = df["lag1_log"]
from sklearn.metrics import mean_absolute_error
baseline_model_score = mean_absolute_error(df["Emissions_log"] , df["baseline_log"])
print(baseline_model_score)
print(f"The baseline model score is {baseline_model_score}. If other models can't beat it, then they are useless.")


#### Model Building

In [ ]:
model_lin = LinearRegression()
model_lin.fit(X, y)

linear_emissions_pred_log = model_lin.predict(X)
linear_emissions_pred = np.exp(linear_emissions_pred_log)

In [ ]:
model_rand = RandomForestRegressor(n_estimators=100, random_state=42)
model_rand.fit(X, y)

random_forest_emissions_pred_log = model_rand.predict(X)
random_forest_emissions_pred = np.exp(random_forest_emissions_pred_log)

In [ ]:
X

In [ ]:
linear_abs_mean = mean_absolute_error(df["Emissions"] , linear_emissions_pred)
random_abs_mean = mean_absolute_error(df["Emissions"] , random_forest_emissions_pred)

In [ ]:
print(f"liner MAE : {linear_abs_mean}")
print(f"Random forest MAE : {random_abs_mean}")

#### Model prediction comparision(Past)

In [ ]:
df["linear_emission_pred"] = linear_emissions_pred
df["random_emission_pred"] = random_forest_emissions_pred

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["Year"],df["Emissions"], label="Actual Emissions", marker='o',
          color='black', linestyle='-', linewidth=2, markersize=6, markerfacecolor='red', markeredgecolor='white', markeredgewidth=1)
plt.plot(df["Year"],df["linear_emission_pred"], label="Linear Regression Prediction", marker='*',
          color= "navy", linestyle='--', linewidth=2, markersize=10, markerfacecolor='skyblue', markeredgecolor='white', markeredgewidth=1)
plt.plot(df["Year"],df["random_emission_pred"], label="Random Forest Prediction", marker='D',
          color='green', linestyle='-.', linewidth=2, markersize=6, markerfacecolor='lime', markeredgecolor='white', markeredgewidth=1)
plt.title("Actual vs Predicted Emissions (1900-2020)" , color = "black" , fontsize = 14 , fontweight = "bold" , fontfamily = "serif")
plt.xlabel("Year")
plt.ylabel("Emissions (Million Metric Tons)")
plt.grid(True)
plt.legend()
plt.show()


#### Predicting future Co2 Emissions

In [ ]:

future_years = [2023,2030,2040,2050,2060]

recursive_forecasting = {
    "year" : [],
    "recursive linear pred" : [],
    "recursive random forest pred" : [],
}
lin_lag1 = df["Emissions_log"].iloc[-2]
lin_lag2 = df["Emissions_log"].iloc[-3]
#print(lin_lag1 , lin_lag2)
rand_lag1 = lin_lag1
rand_lag2 = lin_lag2

for year in future_years:
    lin_growth_rate = lin_lag1 - lin_lag2
    rand_growth_rate = rand_lag1 - rand_lag2
    year_input = year -min_year

    lin_pred_log = model_lin.predict([[lin_lag1 , lin_growth_rate , year_input]])[0]
    #print(lin_lag1 , lin_growth_rate , year , lin_pred_log)
    rand_pred_log = model_rand.predict([[rand_lag1 , rand_growth_rate , year_input]])[0]
    #print(rand_lag1 , rand_growth_rate , year , rand_pred_log)
    lin_pred = np.exp(lin_pred_log)
    rand_pred = np.exp(rand_pred_log) 
    #print(lin_pred , rand_pred)

    recursive_forecasting["year"].append(year)
    recursive_forecasting["recursive linear pred"].append(lin_pred)
    recursive_forecasting["recursive random forest pred"].append(rand_pred)

    lin_lag2 = lin_lag1
    lin_lag1 = lin_pred_log

    rand_lag2 = rand_lag1
    rand_lag1 = rand_pred_log

future_recursive_df = pd.DataFrame(recursive_forecasting)
future_recursive_df

In [ ]:
future_recursive_df["percent change linear"] = future_recursive_df["recursive linear pred"].pct_change() * 100
future_recursive_df["precent change random forest"] = future_recursive_df["recursive random forest pred"].pct_change() * 100

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Actual Emissions
axes[0].plot(
    df["Year"], df["Emissions"],
    label="Actual Emission",
    marker="o", color="black", markersize=6, linewidth=2,
    linestyle="-", markerfacecolor="red",
    markeredgecolor="white", markeredgewidth=1
)

# Linear Train Prediction
axes[0].plot(
    df["Year"], df["linear_emission_pred"],
    label="Linear Train Fit",
    color="blue", linestyle=":", linewidth=2
)

# Linear Future Prediction
axes[0].plot(
    future_recursive_df["year"],
    future_recursive_df["recursive linear pred"],
    label="Linear Forecast",
    marker="*", color="navy", markersize=6,
    linewidth=2, linestyle="--",
    markerfacecolor="skyblue", markeredgecolor="white", markeredgewidth=0
)

# Random Forest Train Prediction
axes[0].plot(
    df["Year"], df["random_emission_pred"],
    label="RF Train Fit",
    color="green", linestyle=":", linewidth=2
)

# Random Forest Future Prediction
axes[0].plot(
    future_recursive_df["year"],
    future_recursive_df["recursive random forest pred"],
    label="RF Forecast",
    marker='D', color='darkgreen', linestyle='-.',
    linewidth=2, markersize=6,
    markerfacecolor='lime', markeredgecolor='white', markeredgewidth=1
)

axes[0].set_title(
    "Global CO2 Emission Prediction (Train + Forecast)",
    color="black", fontsize=14, fontweight="bold", fontfamily="serif"
)
axes[0].set_ylabel("Emissions (Million Metric Tons)")
axes[0].set_xlabel("Year")
axes[0].legend()
axes[0].grid(alpha=0.3)


# Actual Percent Change
axes[1].plot(
    df["Year"], df["percent_change"],
    label="Percent Change",
    marker="o", color="black",
    linestyle="-", linewidth=2, markersize=6,
    markerfacecolor="red", markeredgecolor="white", markeredgewidth=1
)

# Linear Percent Change Forecast
axes[1].plot(
    future_recursive_df["year"],
    future_recursive_df["percent change linear"],
    label="Percent Change (Linear)",
    marker="*", color="navy", markersize=6,
    linewidth=2, linestyle="--",
    markerfacecolor="skyblue", markeredgecolor="white", markeredgewidth=0
)

# RF Percent Change Forecast
axes[1].plot(
    future_recursive_df["year"],
    future_recursive_df["precent change random forest"],
    label="Percent Change (RF)",
    marker='D', color='darkgreen', linestyle='-.',
    linewidth=2, markersize=6,
    markerfacecolor='lime', markeredgecolor='white', markeredgewidth=1
)

axes[1].set_title(
    "Percent Change in Global CO2 Emissions",
    color="black", fontsize=14, fontweight="bold", fontfamily="serif"
)
axes[1].set_ylabel("Percent Change")
axes[1].set_xlabel("Year")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()